# Fitness Survey OCR Pipeline (Gemini)
_by Hoang Dung

Local-only Jupyter notebook. No Colab / Google Drive dependency.

-> Need to self-config for successfully running with Google Colab


In [ ]:
# !pip install -q pymupdf pillow openpyxl pandas google-generativeai
!python -m pip install ipykernel pymupdf pillow openpyxl pandas google-generativeai


In [ ]:
import os

PDF_INPUT_DIR = "./input_trunghoc_pdfs"
OUT_DIR = "./ocr_trunghoc_output"
DPI = 300

# Please set your Gemini API key in the environment variable GEMINI_API_KEY before running this script OR set it here directly (not recommended for security reasons).

DEFAULT_GEMINI_API_KEY = ""  # <- Replace with your actual Gemini API key if you want to hardcode it (not recommended).
GEMINI_API_KEY = ""
MODEL_NAME = "gemini-3-flash-preview"  # <- Instead of "gemini-2.5-flash" with the same pricing [Dung confirmed, or you can check more on https://ai.google.dev/gemini-api/docs/pricing]! You can also use "gemini-3.5-turbo", "gemini-3.5-pro", etc,. for better performance, but it costs more.

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PDF_INPUT_DIR, exist_ok=True)

if not GEMINI_API_KEY:
    print("Set GEMINI_API_KEY before running (see README).")
print(f"Model: {MODEL_NAME}")


In [ ]:
# All 87 output fields, in fixed order.
COLUMN_NAMES = [
    "学校名", "性別", "No.", "握力_右", "握力_左", "上体起こし", "長座体前屈", "反復横とび",
    "持久走_分", "持久走_秒", "20mシャトルラン", "50m走", "立ち幅とび", "ハンドボール投げ", "身長", "体重",
    "質1", "質2-❶", "質2-❷", "質2-❸", "質2-❹", "質2-❺", "質3",
    "質4-❶", "質4-❷", "質4-❸", "質4-❹", "質4-❺", "質4-❻その他", "質5",
    "質6-❶", "質6-❷", "質6-❸", "質7-①", "質7-②", "質7-③", "質7-④", "質7-⑤",
    "質8_部活動_月", "質8_部活動_火", "質8_部活動_水", "質8_部活動_木", "質8_部活動_金", "質8_部活動_土", "質8_部活動_日",
    "質8_地域_月", "質8_地域_火", "質8_地域_水", "質8_地域_木", "質8_地域_金", "質8_地域_土", "質8_地域_日",
    "質8_それ以外_月", "質8_それ以外_火", "質8_それ以外_水", "質8_それ以外_木", "質8_それ以外_金", "質8_それ以外_土", "質8_それ以外_日",
    "質8-2-①", "質8-2-②", "質8-2-③", "質8-2-④", "質8-2-⑤", "質8-2-⑤その他",
    "質9", "質10", "質11", "質12",
    "質12-2-①", "質12-2-②", "質12-2-③", "質12-2-④", "質12-2-⑤", "質12-2-⑥", "質12-2-⑦", "質12-2-⑧", "質12-2-⑨", "質12-2-⑩", "質12-2-⑩その他",
    "質13", "質14", "質15", "質16", "質17", "質18", "質19"
]
assert len(COLUMN_NAMES) == 87

# ROI groups matching physical form layout.
# Page = one spread: LEFT half (質8~19), RIGHT half (header + 実技 + 質1~7).
GROUP_HEADER_MEASURE = ["学校名","性別","No.","握力_右","握力_左","上体起こし","長座体前屈","反復横とび",
                         "持久走_分","持久走_秒","20mシャトルラン","50m走","立ち幅とび","ハンドボール投げ","身長","体重"]

GROUP_SURVEY_1_7 = ["質1","質2-❶","質2-❷","質2-❸","質2-❹","質2-❺","質3","質4-❶","質4-❷","質4-❸",
                     "質4-❹","質4-❺","質4-❻その他","質5","質6-❶","質6-❷","質6-❸",
                     "質7-①","質7-②","質7-③","質7-④","質7-⑤"]

GROUP_CLUB_TIME = ["質8_部活動_月","質8_部活動_火","質8_部活動_水","質8_部活動_木","質8_部活動_金","質8_部活動_土","質8_部活動_日",
                    "質8_地域_月","質8_地域_火","質8_地域_水","質8_地域_木","質8_地域_金","質8_地域_土","質8_地域_日",
                    "質8_それ以外_月","質8_それ以外_火","質8_それ以外_水","質8_それ以外_木","質8_それ以外_金","質8_それ以外_土","質8_それ以外_日",
                    "質8-2-①","質8-2-②","質8-2-③","質8-2-④","質8-2-⑤","質8-2-⑤その他"]

GROUP_DAILY_HABIT = ["質9","質10","質11"]

GROUP_HEALTH_CLASS = ["質12","質12-2-①","質12-2-②","質12-2-③","質12-2-④","質12-2-⑤","質12-2-⑥",
                       "質12-2-⑦","質12-2-⑧","質12-2-⑨","質12-2-⑩","質12-2-⑩その他",
                       "質13","質14","質15","質16","質17","質18","質19"]

ALL_GROUPS = [
    ("header_measure", GROUP_HEADER_MEASURE),
    ("survey_1_7", GROUP_SURVEY_1_7),
    ("club_time", GROUP_CLUB_TIME),
    ("daily_habit", GROUP_DAILY_HABIT),
    ("health_class", GROUP_HEALTH_CLASS),
]
assert sorted(sum([g for _, g in ALL_GROUPS], [])) == sorted(COLUMN_NAMES)

# ROI boundaries as fraction of each half-page height. Adjust if your scan layout differs.
ROI_FRACTIONS = {
    "header_measure": ("right", 0.00, 0.325),
    "survey_1_7":      ("right", 0.285, 1.00),
    "club_time":       ("left",  0.00, 0.29),
    "daily_habit":     ("left",  0.245, 0.435),
    "health_class":    ("left",  0.375, 1.00),
}

# FIELD_HINTS: real printed question text bound to each field key, so the model
# anchors on form content instead of guessing from surrounding rows/columns.
# Groups "質7", "質8-2", "質12-2" use a multi-column checkbox layout where the
# printed numbering goes left-to-right per row, then down (row-major) -
# NOT top-to-bottom per column. Mislabeling this order causes selected marks
# to land on the wrong field key.
FIELD_HINTS = {
    "質1": "運動やスポーツをすることは好きですか。選択肢:好き/やや好き/やや嫌い/嫌い",
    "質2-❶": "❶運動やスポーツをすること。選択肢:ある/ややある/あまりない/ない",
    "質2-❷": "❷運動やスポーツをみること",
    "質2-❸": "❸運動やスポーツを支えること(大会運営のボランティアなど)",
    "質2-❹": "❹運動やスポーツを知ること(話を聞く、調べるなど)",
    "質2-❺": "❺運動やスポーツを通して様々な人が集まって交流したり、つながり一体感を感じたりすること",
    "質3": "運動やスポーツをして楽しいと感じますか。選択肢:感じる/やや感じる/あまり感じない/感じない",
    "質4-❶": "❶体を動かしてすっきりした気分になったとき",
    "質4-❷": "❷様々な種目を体験したとき",
    "質4-❸": "❸できなかったことができるようになったとき",
    "質4-❹": "❹記録に挑戦したり、記録が上がったり、競い合ったりしたとき",
    "質4-❺": "❺友達と交流したり、協力できたとき",
    "質4-❻その他": "❻その他(自由記述の手書きテキストをそのまま転記。空欄ならnull)",
    "質5": "卒業後も自主的に運動やスポーツをする時間を持ちたいと思いますか。選択肢:思う/やや思う/あまり思わない/思わない",
    "質6-❶": "❶運動すること。選択肢:ある/ややある/あまりない/ない",
    "質6-❷": "❷スポーツの話をすること",
    "質6-❸": "❸スポーツ観戦をすること(テレビ観戦を含む)",
    # 質7: single row, 5 marks. Note ③ is ONE combined label (スポーツ枠+その他スポーツクラブ), not two.
    "質7-①": "①学校の運動部(0/1)",
    "質7-②": "②学校の文化部(0/1)",
    "質7-③": "③地域クラブ活動(スポーツ)、その他のスポーツクラブ ― 1つの選択肢として扱う(0/1)",
    "質7-④": "④地域クラブ活動(文化)(0/1)",
    "質7-⑤": "⑤所属していない(0/1)",
    # 質8-2: 2-column x 2-row grid + その他, numbered row-major (left→right, then next row).
    "質8-2-①": "①運動する時間がないから(0/1、行1左)",
    "質8-2-②": "②運動する場所がないから(0/1、行1右)",
    "質8-2-③": "③一緒に運動する友達がいないから(0/1、行2左)",
    "質8-2-④": "④運動が好きではないから(0/1、行2右)",
    "質8-2-⑤": "⑤その他(0/1、最終行)。チェックされていれば右の自由記述欄に手書き理由があるはずなので、"
                "その内容は「質8-2-⑤その他」キーに転記すること",
    "質8-2-⑤その他": "質8-2-⑤「その他」の横に書かれた自由記述欄の手書きテキスト。"
                    "⑤が0(未チェック)、または欄が空欄の場合はnull",
    "質12": "保健体育の授業は楽しいですか。選択肢:楽しい/やや楽しい/あまり楽しくない/楽しくない",
    # 質12-2: 2-column x 5-row grid + その他, numbered row-major (left→right, then next row).
    "質12-2-①": "①運動のポイントを分かりやすく教えてもらえたら(0/1、行1左)",
    "質12-2-②": "②できなかったことができるようになったら(0/1、行1右)",
    "質12-2-③": "③自分に合った場やルールが用意されていたら(0/1、行2左)",
    "質12-2-④": "④タブレットなどのICTを活用できたら(0/1、行2右)",
    "質12-2-⑤": "⑤先生にほめてもらえたら(0/1、行3左)",
    "質12-2-⑥": "⑥友達に認めてもらえたら(0/1、行3右)",
    "質12-2-⑦": "⑦先生に個別に指導してもらえたら(0/1、行4左)",
    "質12-2-⑧": "⑧自分に合ったペースで行うことができたら(0/1、行4右)",
    "質12-2-⑨": "⑨できる・できないだけで比較されなかったら(0/1、行5左、右列なし)",
    "質12-2-⑩": "⑩その他(0/1、最終行)。チェックされていれば横の自由記述欄に手書き内容があるはずなので、"
                "その内容は「質12-2-⑩その他」キーに転記すること",
    "質12-2-⑩その他": "質12-2-⑩「その他」の横に書かれた自由記述欄の手書きテキスト。"
                     "⑩が0(未チェック)、または欄が空欄の場合はnull",
    "質13": "自分から「やってみたい」と思うときがありますか。選択肢:いつもある/だいたいある/あまりない/全くない",
    "質14": "目標(ねらい)を意識して学習することがありますか。同じ4択",
    "質15": "友達と助け合ったり、教え合ったりして学習することがありますか。同じ4択",
    "質16": "うまくいかないことがあったときにも、考え話し合って取り組みましたか。選択肢:いつも取り組めていた/だいたい取り組めていた/あまり取り組めていなかった/取り組めていなかった",
    "質17": "タブレットなどのICTを使って学習しましたか。選択肢:いつもある/だいたいある/あまりない/全くない/ICTを活用していない",
    "質18": "運動、食事、休養、睡眠に気をつけた生活を送れていると思いますか。選択肢:思う/やや思う/あまり思わない/思わない",
    "質19": "保健を学習して、もっと運動しようと思いましたか。選択肢:思うようになった/やや思うようになった/あまり思わなかった/思わなかった",
    "反復横とび": "④反復横とび。単位は「点(回)」。手書きの数字のみ(例: 29)",
    "ハンドボール投げ": "⑧ハンドボール投げ。単位は「m」。手書きの数字のみ(例: 4)",
    "身長": "(1)身長。単位は「cm」。小数第1位まで(例: 160.5)",
    "体重": "(2)体重。単位は「kg」。小数第1位まで(例: 56.9)",
    "握力_右": "①握力・右。単位 kg",
    "握力_左": "①握力・左。単位 kg",
    "上体起こし": "②上体起こし。単位 回",
    "長座体前屈": "③長座体前屈。単位 cm",
    "50m走": "⑥50m走。単位 秒",
    "立ち幅とび": "⑦立ち幅とび。単位 cm",
    "20mシャトルラン": "⑥20mシャトルラン。単位 回",
    "持久走_分": "⑤持久走の「分」。20mシャトルランとの選択種目のため、実施していなければ空欄(null)",
    "持久走_秒": "⑤持久走の「秒」。20mシャトルランとの選択種目のため、実施していなければ空欄(null)",
}

# "その他" checkbox fields paired with a free-text field capturing the handwritten note.
TEXT_COMPANION_GROUPS = {"survey_1_7", "club_time", "health_class"}

# Groups containing a multi-column (row-major) checkbox grid.
GRID_CHECKLIST_GROUPS = {"club_time", "health_class"}


In [ ]:
import fitz
import numpy as np
from PIL import Image

def render_page(pdf_path, page_num, dpi=DPI):
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    zoom = dpi / 72
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    return img

def crop_roi(full_page_img, side, frac_top, frac_bottom):
    """side: 'left' or 'right' half of the page; frac_top/bottom: height ratio (0-1)."""
    w, h = full_page_img.size
    x0, x1 = (0, w // 2) if side == "left" else (w // 2, w)
    y0, y1 = int(h * frac_top), int(h * frac_bottom)
    return full_page_img.crop((x0, y0, x1, y1))

def get_all_rois(full_page_img):
    rois = {}
    for group_name, (side, top, bot) in ROI_FRACTIONS.items():
        rois[group_name] = crop_roi(full_page_img, side, top, bot)
    return rois


In [ ]:
import io, json, time
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

def build_schema(fields):
    return {
        "type": "object",
        "properties": {f: {"type": "string", "nullable": True} for f in fields},
        "required": fields,
    }

def build_prompt(fields, group_name):
    hint_lines = [f'- "{f}": {FIELD_HINTS[f]}' for f in fields if f in FIELD_HINTS]
    hint_block = ""
    if hint_lines:
        hint_block = "各キーが対応する設問文（フォーム上の印刷文）:\n" + "\n".join(hint_lines) + "\n"

    anti_shift = ""
    if group_name in ("survey_1_7", "health_class"):
        anti_shift = (
            "【重要・厳守】各設問は横一列に4つの選択肢が並んでいます。"
            "必ず「この設問番号の行」に実際に塗られている／マークされている選択肢だけを読み取り、"
            "前後の設問のパターンから類推して埋めないでください。設問ごとに独立して判定してください。\n"
        )

    grid_order = ""
    if group_name in GRID_CHECKLIST_GROUPS:
        grid_order = (
            "【重要・厳守】チェックリスト形式の設問(質8-2、質12-2)は2列レイアウトです。"
            "番号は「左列から右列へ、その後次の行へ」の順（行優先）で振られています。"
            "列ごとに上から下へ読み進めないでください。各キーに紐づく設問文(上記)と"
            "画像上の実際の印刷テキストを照合し、正しい行・列のチェック有無だけを読み取ってください。\n"
        )

    anti_overmark = ""
    if group_name == "survey_1_7":
        anti_overmark = (
            "質7は複数選択可能なチェック欄ですが、実際にはっきりと黒い点／塗りつぶしがある項目のみ1、"
            "それ以外は全て0としてください。薄い印刷線や枠を選択マークと誤認しないでください。\n"
        )

    text_companion = ""
    if group_name in TEXT_COMPANION_GROUPS:
        text_companion = (
            "【重要】「その他」のチェック欄の横または下には、手書きの自由記述欄があります。"
            "対応するチェックが1(あり)の場合、その手書きテキストを対応する「...その他」キーに"
            "そのまま転記してください。小さい文字も見落とさず確認してください。"
            "チェックが0、または記述欄が空欄の場合はnullとしてください。\n"
        )

    return f"""あなたはOCRのプロフェッショナルです。添付画像から次のキーの値を正確に読み取り、JSONのみ返してください。
読み取れない・空欄は null。
{hint_block}{anti_shift}{grid_order}{anti_overmark}{text_companion}
キー一覧: {json.dumps(fields, ensure_ascii=False)}"""

def call_gemini_group(pil_image, fields, group_name, temperature=0.0):
    model = genai.GenerativeModel(MODEL_NAME)
    prompt = build_prompt(fields, group_name)
    
    resp = model.generate_content(
        [prompt, pil_image],
        generation_config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_schema": build_schema(fields),
        },
        request_options={"timeout": 2000} 
    )
    
    # time.sleep(5)  # basic rate-limit throttle
    return json.loads(resp.text)

def call_group_self_consistent(pil_image, fields, group_name):
    """Single-call mode (self-consistency double-check disabled to save cost)."""
    r1 = call_gemini_group(pil_image, fields, group_name, temperature=0.0)
    final, flags = {}, {}
    for f in fields:
        final[f] = r1.get(f)
        flags[f] = "OK"
    return final, flags


In [ ]:
import os
import pickle
import time
from pathlib import Path

CHECKPOINT_DIR = os.path.join(OUT_DIR, "checkpoints")
GROUP_CHECKPOINT_DIR = os.path.join(OUT_DIR, "group_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(GROUP_CHECKPOINT_DIR, exist_ok=True)

def _safe_stem(pdf_path: str) -> str:
    return Path(pdf_path).stem.replace("/", "_").replace("\\", "_")

def checkpoint_path(pdf_path: str, page_num: int) -> str:
    return os.path.join(CHECKPOINT_DIR, f"{_safe_stem(pdf_path)}_p{page_num:04d}.pkl")

def save_page_checkpoint(pdf_path: str, page_num: int, record: dict, flag_record: dict):
    path = checkpoint_path(pdf_path, page_num)
    tmp_path = path + ".tmp"
    
    # Ghi vào file tạm trước để tránh lỗi corrupted nếu bị ngắt đột ngột
    with open(tmp_path, "wb") as f:
        pickle.dump(
            {
                "pdf": os.path.basename(pdf_path),
                "page": page_num,
                "record": record,
                "flags": flag_record,
            },
            f,
        )
    # Đổi tên file tạm thành file chính (thao tác này là atomic của HĐH nên tuyệt đối an toàn)
    os.replace(tmp_path, path)

def group_checkpoint_path(pdf_path: str, page_num: int, group_name: str) -> str:
    return os.path.join(GROUP_CHECKPOINT_DIR, f"{_safe_stem(pdf_path)}_p{page_num:04d}_{group_name}.pkl")

def save_group_checkpoint(pdf_path: str, page_num: int, group_name: str, final: dict, flags: dict):
    path = group_checkpoint_path(pdf_path, page_num, group_name)
    tmp_path = path + ".tmp"
    with open(tmp_path, "wb") as f:
        pickle.dump({"final": final, "flags": flags}, f)
    os.replace(tmp_path, path)

def load_group_checkpoint(pdf_path: str, page_num: int, group_name: str):
    path = group_checkpoint_path(pdf_path, page_num, group_name)
    if not os.path.exists(path):
        return None
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except:
        return None

def load_page_checkpoint(pdf_path: str, page_num: int):
    path = checkpoint_path(pdf_path, page_num)
    if not os.path.exists(path):
        return None
    
    # Bổ sung try-except chống sập nếu file pickle bị lỗi định dạng
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except (EOFError, pickle.UnpicklingError, Exception) as e:
        print(f"      ⚠ Checkpoint bị lỗi cho page {page_num} ({e}). Sẽ tiến hành xử lý lại.")
        return None

def is_retryable_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    name = type(exc).__name__.lower()
    return (
        "deadline" in msg
        or "deadlineexceeded" in name
        or "504" in msg
        or "503" in msg
        or "500" in msg
        or "429" in msg
        or "resourceexhausted" in name
        or "quota" in name
        or "serviceunavailable" in name
        or "internalservererror" in name
        or "stream removed" in msg
        or "timeout" in msg
        or "timed out" in msg
        or "connection" in msg
    )

def call_with_retry(pil_image, fields, group_name, max_retries=4, base_wait=10):
    """Gọi OCR 1 group; retry khi gặp lỗi mạng, server, hoặc bị rate limit (429)."""
    last_err = None
    for attempt in range(max_retries):
        try:
            return call_group_self_consistent(pil_image, fields, group_name)
        except Exception as e:
            last_err = e
            if is_retryable_error(e) and attempt < max_retries - 1:
                wait = base_wait * (2 ** attempt)
                print(f"      ⚠ API Error ({type(e).__name__}), retry {attempt+1}/{max_retries} sau {wait}s...")
                time.sleep(wait)
                continue
            raise
    if last_err is not None:
        raise last_err
    raise RuntimeError("Failed to execute call_with_retry, max_retries might be <= 0")


## Run pipeline on all PDFs in `PDF_INPUT_DIR`

In [ ]:
import glob

pdf_files = sorted(glob.glob(os.path.join(PDF_INPUT_DIR, "*.pdf")))
print(f"Found {len(pdf_files)} PDF file(s) in {PDF_INPUT_DIR}")

all_final, all_flags_rows = [], []
n_skipped, n_done, n_failed = 0, 0, 0

for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    doc.close()
    print(f"\n{os.path.basename(pdf_path)} - {n_pages} page(s)")

    for page_num in range(n_pages):
        # ---- Resume từ checkpoint ----
        cached = load_page_checkpoint(pdf_path, page_num)
        if cached is not None:
            all_final.append(cached["record"])
            all_flags_rows.append(cached["flags"])
            n_skipped += 1
            print(f"   page {page_num}: SKIP (cached) No.={cached['record'].get('No.')}")
            continue

        # ---- OCR mới ----
        t0 = time.time()
        try:
            full_img = render_page(pdf_path, page_num)
            rois = get_all_rois(full_img)

            record, flag_record = {}, {}
            for group_name, fields in ALL_GROUPS:
                cached_group = load_group_checkpoint(pdf_path, page_num, group_name)
                if cached_group is not None:
                    final, flags = cached_group["final"], cached_group["flags"]
                else:
                    final, flags = call_with_retry(rois[group_name], fields, group_name)
                    save_group_checkpoint(pdf_path, page_num, group_name, final, flags)
                record.update(final)
                flag_record.update(flags)

            # Lưu ngay sau khi xong 1 page
            save_page_checkpoint(pdf_path, page_num, record, flag_record)

            # Dọn dẹp các checkpoint tạm của group sau khi đã lưu xong page checkpoint
            for group_name, _ in ALL_GROUPS:
                g_path = group_checkpoint_path(pdf_path, page_num, group_name)
                if os.path.exists(g_path):
                    try: os.remove(g_path)
                    except: pass

            all_final.append(record)
            all_flags_rows.append(flag_record)
            n_done += 1
            print(f"   page {page_num}: No.={record.get('No.')} ({time.time()-t0:.1f}s)")

        except Exception as e:
            n_failed += 1
            print(f"   page {page_num}: FAILED — {type(e).__name__}: {e}")
            # Không append vào all_final để export không bị lệch;
            # page này sẽ được thử lại ở lần chạy sau (vì chưa có checkpoint).
            # Chèn một dòng lỗi vào data để giữ nguyên vị trí dòng trong Excel (Chống sót file)
            empty_record = {col: f"ERROR: {type(e).__name__}" if col == "学校名" else None for col in COLUMN_NAMES}
            all_final.append(empty_record)
            all_flags_rows.append({col: "FAILED" for col in COLUMN_NAMES})
            continue

print(f"\nDone. total={len(all_final)} | new={n_done} | skipped={n_skipped} | failed={n_failed}")

## Export results to Excel

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font

if not all_final:
    raise RuntimeError("all_final rỗng — không có page nào thành công / checkpoint.")

timestamp = time.strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(OUT_DIR, f"体力測定調査結果_{MODEL_NAME}_{timestamp}.xlsx")
# blue: đổi tên file nếu cần, ví dụ bỏ _小学校_

df_final = pd.DataFrame(all_final).reindex(columns=COLUMN_NAMES)
df_flags = pd.DataFrame(all_flags_rows).reindex(columns=COLUMN_NAMES)

wb = openpyxl.Workbook()
wb.remove(wb.active)

def append_df(wb, df, name):
    ws = wb.create_sheet(title=name)
    ws.append(list(df.columns))
    for row in df.fillna("").values.tolist():
        ws.append(row)
    return ws

ws_final = append_df(wb, df_final, "納品データ")
ws_flags = append_df(wb, df_flags, "自己一致性チェック")

red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
red_font = Font(color="9C0006", bold=True)
for r in range(2, len(df_final) + 2):
    for c, col in enumerate(COLUMN_NAMES, start=1):
        if ws_flags.cell(row=r, column=c).value == "MISMATCH":
            cell = ws_final.cell(row=r, column=c)
            cell.fill = red_fill
            cell.font = red_font

wb.save(out_path)
print(f"Saved: {out_path}")